In [1]:
import os
import re
import json
import glob
import subprocess
import torch
import numpy as np
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("OpenGVLab/InternVL3-8B-hf")

config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.86k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [2]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
TYPES = ["attribute", "relation"]
CAND_IDS = [9454, 2753, 9693, 2152]
MEASURED_SEC_PER_QUESTION = 5.46

CACHE_DIRS = {
    "attribute": "/kaggle/input/datasets/ponishbhatia2/internvl3-attribute-questionaware",
    "relation": "/kaggle/input/datasets/ponishbhatia2/question-aware-cache-internvl3-relation",
}

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
expected = {
    t: {q["question_id"] for q in all_questions if q["hallucination_type"] == t and q["answer_format"] == "yes_no"}
    for t in TYPES
}
print(len(all_questions), "questions loaded,", {t: len(v) for t, v in expected.items()}, "yes/no questions")
print(tokenizer.convert_ids_to_tokens(CAND_IDS))
assert tokenizer.convert_ids_to_tokens(CAND_IDS) == ["Yes", "No", "yes", "no"]

with open(f"{INPUT}/vlm-labeled-examples/labeled_examples.json") as f:
    labels = pd.DataFrame(json.load(f))
assert "cache" in labels.columns, "attach vlm-labeled-examples version 3"
split_of = labels[labels["cache"] == "question_aware"].set_index("question_id")["split"].to_dict()
print(len(split_of), "questions with a split")

17492 questions loaded, {'attribute': 7628, 'relation': 6506} yes/no questions
['Yes', 'No', 'yes', 'no']
17126 questions with a split


In [3]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


def cache_files(t):
    files = sorted(glob.glob(f"{CACHE_DIRS[t]}/**/{t}_batch_*.pt", recursive=True), key=batch_number)
    assert files, f"no {t}_batch files under {CACHE_DIRS[t]}"
    return files


records = []
for t in TYPES:
    files = cache_files(t)
    for path in files:
        batch = torch.load(path, mmap=True)
        for e in batch:
            records.append({
                "question_id": e["question_id"],
                "type": t,
                "prefix": e["question_id"].split("_")[0] + "_",
                "top_id": int(e["answer_logits"].argmax()),
                "answer_text": e["answer_text"],
                "n_image_tokens": e["n_image_tokens"],
                "cand": e["answer_logits"][CAND_IDS].float().tolist(),
            })
        del batch
    ids = {r["question_id"] for r in records if r["type"] == t}
    print(f"{t}: {len(files)} files, {len(ids)} entries, coverage {len(ids & expected[t]) / len(expected[t]):.4f}")
    assert len(ids & expected[t]) > 0.95 * len(expected[t]), f"{t} cache looks incomplete"

rec = pd.DataFrame(records)
rec = rec.drop_duplicates("question_id").reset_index(drop=True)

YES_IDS, NO_IDS = {}, {}
for prefix in ["amber_", "reefknot_"]:
    sub = rec[rec.prefix == prefix]
    lower = sub.answer_text.str.strip().str.lower()
    for name, store in [("yes", YES_IDS), ("no", NO_IDS)]:
        share = sub[lower == name].top_id.value_counts(normalize=True)
        keep = [int(i) for i, v in share.items() if v >= 0.1]
        assert keep and all(i in CAND_IDS for i in keep), f"unexpected {name} tokens for {prefix}"
        assert share[keep].sum() > 0.95, f"{name} answers for {prefix} spread over too many tokens"
        store[prefix] = keep
        print(f"{prefix:10s} {name:3s} " + " | ".join(f"{tokenizer.decode(i)!r} {v:.3f}" for i, v in share.head(4).items()) + f" -> using {keep}")


def gap_of(cand, prefix):
    c = dict(zip(CAND_IDS, cand))
    return np.mean([c[i] for i in YES_IDS[prefix]]) - np.mean([c[i] for i in NO_IDS[prefix]])


rec["gap"] = [gap_of(c, p) for c, p in zip(rec.cand, rec.prefix)]
rec["in_pair"] = [t in YES_IDS[p] + NO_IDS[p] for t, p in zip(rec.top_id, rec.prefix)]
rec["source"] = rec.prefix.str.rstrip("_")
rec["truth"] = rec.question_id.map(lambda q: question_by_id[q]["ground_truth_answer"].strip().lower())
rec["model_answer"] = np.where(rec.gap > 0, "yes", "no")
rec["group"] = np.select(
    [
        (rec.truth == "yes") & (rec.model_answer == "yes"),
        (rec.truth == "no") & (rec.model_answer == "no"),
        (rec.truth == "no") & (rec.model_answer == "yes"),
        (rec.truth == "yes") & (rec.model_answer == "no"),
    ],
    ["true_yes", "true_no", "false_yes", "false_no"],
    default="other",
)
rec["split"] = rec.question_id.map(split_of)
print(rec.split.isna().sum(), "cached questions without a split")
rec = rec[rec.split.notna()].reset_index(drop=True)

print(rec.groupby(["type", "source"]).in_pair.agg(["mean", "size"]).round(4))
print(pd.crosstab([rec.type, rec.source, rec.in_pair], rec.group))
print("non-pair top-1 tokens:", Counter(tokenizer.decode(i) for i in rec[~rec.in_pair].top_id).most_common(8))

attribute: 31 files, 7628 entries, coverage 1.0000
relation: 27 files, 6506 entries, coverage 1.0000
amber_     yes 'Yes' 1.000 -> using [9454]
amber_     no  'No' 1.000 -> using [2753]
reefknot_  yes 'yes' 0.516 | 'Yes' 0.484 -> using [9693, 9454]
reefknot_  no  'No' 0.576 | 'no' 0.424 -> using [2753, 2152]
8 cached questions without a split
                      mean  size
type      source                
attribute amber     0.7682  7620
relation  amber     0.7951  1664
          reefknot  1.0000  4842
group                       false_no  false_yes  true_no  true_yes
type      source   in_pair                                        
attribute amber    False         152        205      813       596
                   True          229        413     2378      2834
relation  amber    False          89          0      191        61
                   True          182          8      490       643
          reefknot True          259       1297     1124      2162
non-pair top-1 tokens

In [4]:
TARGETS = {"attribute": 3000, "relation": 3500}
MIN_BUDGET = 1000
N_NONPAIR = 150
SHARES = {"true_yes": 0.6, "true_no": 0.4}

parts = []
for t in TYPES:
    d = rec[(rec.type == t) & rec.in_pair]
    fy = d[d.group == "false_yes"]
    budget = max(TARGETS[t] - len(fy), MIN_BUDGET)
    parts.append(fy.assign(stratum="false_yes"))
    for g, share in SHARES.items():
        pool = d[d.group == g]
        if len(pool) == 0:
            continue
        quota = min(int(budget * share), len(pool))
        parts.append(pool.groupby(["source", "split"], group_keys=False).sample(frac=quota / len(pool), random_state=0).assign(stratum=g))
    pool = rec[(rec.type == t) & ~rec.in_pair]
    parts.append(pool[pool.group == "false_yes"].assign(stratum="non_pair_false_yes"))
    rest = pool[pool.group != "false_yes"]
    parts.append(rest.sample(n=min(N_NONPAIR, len(rest)), random_state=0).assign(stratum="non_pair"))

selection = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
assert selection.question_id.is_unique
n_false_yes = int(((rec.in_pair) & (rec.group == "false_yes")).sum())
assert (selection.stratum == "false_yes").sum() == n_false_yes, "false_yes rows were dropped"

print(pd.crosstab([selection.type, selection.source, selection.split], selection.stratum, margins=True))
sel_counts = selection.groupby("type").size().to_dict()
print("selected per type:", sel_counts, "of", {t: len(expected[t]) for t in TYPES})
print(f"{len(selection)} questions at {MEASURED_SEC_PER_QUESTION} s/question: about {len(selection) * MEASURED_SEC_PER_QUESTION / 3600:.1f} h of collection")

stratum                   false_yes  non_pair  non_pair_false_yes  true_no  \
type      source   split                                                     
attribute amber    test          72        25                  18      159   
                   train        284       111                 159      717   
                   val           57        14                  28      158   
relation  amber    test           1        21                   0       41   
                   train          4       104                   0      182   
                   val            3        25                   0       44   
          reefknot test         188         0                   0       97   
                   train        905         0                   0      424   
                   val          204         0                   0       90   
All                            1718       300                 205     1912   

stratum                   true_yes   All  
type      source   s

In [5]:
SEL_DIR = "/kaggle/working/dla_intern_selection"
os.makedirs(SEL_DIR, exist_ok=True)

with open(os.path.join(SEL_DIR, "selection.json"), "w") as f:
    json.dump({
        "yes_ids": YES_IDS,
        "no_ids": NO_IDS,
        "cand_ids": CAND_IDS,
        "selected": {t: selection[selection.type == t].question_id.tolist() for t in TYPES},
        "stratum": dict(zip(selection.question_id, selection.stratum)),
        "sec_per_question": MEASURED_SEC_PER_QUESTION,
    }, f)

with open(os.path.join(SEL_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dla-intern-selection",
        "id": f"{USERNAME}/dla-intern-selection",
        "subtitle": "Question subsample for InternVL3 direct logit attribution",
        "description": "Questions selected for InternVL3-8B direct logit attribution from the question-aware caches: every false_yes row whose "
                       "top-1 is one of the yes/no tokens, a stratified sample of true_yes (60%) and true_no (40%) controls by source and split, "
                       "every false_yes row whose top-1 is outside the yes/no tokens plus a small sample of the other non-pair rows for a "
                       "sensitivity check. stratum maps each question_id to its group.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


ok, out = _run(["kaggle", "datasets", "version", "-p", SEL_DIR, "-m", "v2: larger true_yes and true_no controls, no false_no sample"])
print(out)
assert ok, "dataset version upload failed"

Starting upload for file selection.json
Upload successful: selection.json (322KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/nocturnalnerd18/dla-intern-selection

  0%|          | 0.00/322k [00:00<?, ?B/s]
100%|██████████| 322k/322k [00:00<00:00, 471kB/s]
